In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from __future__ import annotations

import json
import re
import tarfile
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse

import requests
from lxml import etree

In [3]:
# información de contacto para la API de NCBI
EMAIL = "diegoagudelo30@gmail.com"
TOOL_NAME = "diet-extractor-demo/0.1"

# Ruta al Excel con los registros
EXCEL_PATH = Path("tabla_estudios.xlsx")

# Directorio raíz del proyecto
PROJECT_DIR = Path(".").resolve()

# Segundos de espera entre llamadas a la API de NCBI para evitar bloqueos por exceso de solicitudes
NCBI_SLEEP = 0.5

In [4]:
# funciones utilitarias
def pmcid_from_fultxt_filename(s: str) -> Optional[str]:
    if not s:
        return None
    m = re.search(r"(PMC\d+)", str(s))
    return m.group(1) if m else None


def safe_write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def ftp_to_https(url: str) -> str:
    if url.startswith("ftp://ftp.ncbi.nlm.nih.gov/"):
        return url.replace("ftp://ftp.ncbi.nlm.nih.gov/", "https://ftp.ncbi.nlm.nih.gov/")
    return url


def download_file(url: str, out_path: Path, timeout=60) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path

    headers = {"User-Agent": f"{TOOL_NAME} ({EMAIL})"}
    with requests.get(url, stream=True, timeout=timeout, headers=headers) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return out_path


def untar(tar_path: Path, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(out_dir)
    return out_dir


# ====== OA service (XML) ======
def pmc_oa_links_xml(pmcid: str) -> etree._Element:
    base = "https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi"
    params = {"id": pmcid}
    headers = {"User-Agent": f"{TOOL_NAME} ({EMAIL})"}
    r = requests.get(base, params=params, timeout=60, headers=headers)
    r.raise_for_status()
    return etree.fromstring(r.content)


def oa_pick_links(root: etree._Element) -> dict:
    out = {"tgz": None, "pdf": None}
    for link in root.xpath(".//link"):
        fmt = link.get("format")
        href = link.get("href")
        if not fmt or not href:
            continue
        if fmt.lower() in ("tgz", "tar.gz"):
            out["tgz"] = href
        elif fmt.lower() == "pdf":
            out["pdf"] = href
    return out


# ====== Fallback PMC direct ======
def pmc_pdf_url_guess(pmcid: str) -> str:
    return f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/pdf/"


def pmc_nxml_url_guess(pmcid: str) -> str:
    return f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/bin/{pmcid}.nxml"


def try_download_with_redirect(url: str, out_path: Path) -> bool:
    try:
        headers = {"User-Agent": f"{TOOL_NAME} ({EMAIL})"}
        r = requests.get(url, stream=True, timeout=60, headers=headers, allow_redirects=True)
        if r.status_code >= 400:
            return False
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        return out_path.exists() and out_path.stat().st_size > 0
    except Exception:
        return False


# ====== Core: procesar una fila ======
def run_one(row: dict, project_dir: Path) -> dict:
    pmid = str(row["pubmed_id"])
    pmcid = pmcid_from_fultxt_filename(row.get("fultxt_filename", ""))

    raw_dir = project_dir / "data" / "raw" / pmid
    ext_dir = project_dir / "data" / "extracted" / pmid
    raw_dir.mkdir(parents=True, exist_ok=True)
    ext_dir.mkdir(parents=True, exist_ok=True)

    manifest = {
        "pubmed_id": pmid,
        "pmcid": pmcid,
        "title": row.get("title"),
        "year": row.get("year"),
        "bioproject": row.get("bioproject"),
        "downloads": {},
        "notes": [],
    }

    # 1) OA service (XML)
    tgz_url = None
    pdf_url = None
    if pmcid:
        try:
            root = pmc_oa_links_xml(pmcid)
            links = oa_pick_links(root)
            tgz_url = links["tgz"]
            pdf_url = links["pdf"]
            manifest["downloads"]["oa_service_ok"] = True
            manifest["downloads"]["oa_tgz_url"] = tgz_url
            manifest["downloads"]["oa_pdf_url"] = pdf_url
        except Exception as e:
            manifest["downloads"]["oa_service_ok"] = False
            manifest["notes"].append(f"OA service XML error: {repr(e)}")
    else:
        manifest["notes"].append("No PMCID found from fultxt_filename.")

    # 2) Descargar tgz
    if tgz_url:
        tgz_url_https = ftp_to_https(tgz_url)
        try:
            tgz_path = download_file(tgz_url_https, raw_dir / "pmc_package.tar.gz")
            untar(tgz_path, raw_dir / "pmc_package")
            manifest["downloads"]["pmc_tgz_downloaded"] = True
        except Exception as e:
            manifest["downloads"]["pmc_tgz_downloaded"] = False
            manifest["notes"].append(f"TGZ download/extract error: {repr(e)}")
    else:
        manifest["downloads"]["pmc_tgz_downloaded"] = False
        manifest["notes"].append("No OA tgz link found (paper may not be in OA subset).")

    # 3) PDF: OA link → fallback PMC
    pdf_downloaded = False
    if pdf_url:
        try:
            download_file(ftp_to_https(pdf_url), raw_dir / "main.pdf")
            pdf_downloaded = True
        except Exception as e:
            manifest["notes"].append(f"OA PDF download error: {repr(e)}")

    if not pdf_downloaded and pmcid:
        ok = try_download_with_redirect(pmc_pdf_url_guess(pmcid), raw_dir / "main.pdf")
        if ok:
            pdf_downloaded = True
            manifest["notes"].append("PDF downloaded via PMC fallback /pdf/ endpoint.")
        else:
            manifest["notes"].append("PDF fallback failed from PMC /pdf/ endpoint.")

    manifest["downloads"]["pdf_downloaded"] = pdf_downloaded

    # 4) NXML fallback si no hay tgz
    nxml_downloaded = False
    if not (raw_dir / "pmc_package").exists() and pmcid:
        ok = try_download_with_redirect(pmc_nxml_url_guess(pmcid), raw_dir / "main.nxml")
        if ok:
            nxml_downloaded = True
            manifest["notes"].append("NXML downloaded via PMC fallback /bin/PMCID.nxml endpoint.")
        else:
            manifest["notes"].append("NXML fallback failed from /bin/PMCID.nxml endpoint.")
    manifest["downloads"]["nxml_downloaded"] = nxml_downloaded

    # 5) Suplemento (opcional)
    supp_url = row.get("supp_url")
    if supp_url:
        try:
            name = Path(urlparse(supp_url).path).name or "supplement"
            download_file(supp_url, raw_dir / "supp" / name)
            manifest["downloads"]["supp_downloaded"] = True
        except Exception as e:
            manifest["downloads"]["supp_downloaded"] = False
            manifest["notes"].append(f"Supplement download error: {repr(e)}")
    else:
        manifest["downloads"]["supp_downloaded"] = False

    safe_write_json(raw_dir / "manifest.json", manifest)
    safe_write_json(ext_dir / "status.json", {"pubmed_id": pmid, "pmcid": pmcid})

    return manifest


# ====== Leer Excel ======
def load_rows_from_excel(excel_path: Path) -> list[dict]:
    import openpyxl
    wb = openpyxl.load_workbook(excel_path)
    ws = wb.active
    headers = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]

    rows = []
    for raw_row in ws.iter_rows(min_row=2, values_only=True):
        row_dict = dict(zip(headers, raw_row))
        # Normalizar nombres de columna al esquema esperado por run_one
        row_dict["bioproject"] = row_dict.pop("Bioproject", None)
        row_dict.setdefault("supp_url", None)
        # Saltar filas sin pubmed_id
        if not row_dict.get("pubmed_id"):
            continue
        rows.append(row_dict)
    return rows



In [5]:
# ====== Main ======
if __name__ == "__main__":
    rows = load_rows_from_excel(EXCEL_PATH)
    total = len(rows)
    print(f"📋 {total} registros encontrados en {EXCEL_PATH.name}\n")

    summary = []

    for i, row in enumerate(rows, start=1):
        pmid = str(row["pubmed_id"])
        print(f"[{i}/{total}] PMID {pmid} — {str(row.get('title', ''))[:70]}...")

        try:
            manifest = run_one(row, PROJECT_DIR)
            d = manifest["downloads"]
            print(
                f"  ✓ tgz={d.get('pmc_tgz_downloaded')} "
                f"pdf={d.get('pdf_downloaded')} "
                f"nxml={d.get('nxml_downloaded')}"
            )
            for note in manifest["notes"]:
                print(f"  ⚠ {note}")
            summary.append({"pubmed_id": pmid, "ok": True, **d})
        except Exception as e:
            print(f"  ✗ Error fatal: {repr(e)}")
            summary.append({"pubmed_id": pmid, "ok": False, "error": repr(e)})

        # Respetar rate limit de NCBI
        time.sleep(NCBI_SLEEP)

    # Guardar resumen global
    summary_path = PROJECT_DIR / "data" / "download_summary.json"
    safe_write_json(summary_path, summary)
    print(f"\n✅ Proceso completado. Resumen en: {summary_path}")

📋 12 registros encontrados en tabla_estudios.xlsx

[1/12] PMID 40676527 — Improving genomic prediction accuracy for methane emission and feed ef...
  ✓ tgz=True pdf=True nxml=False
  ⚠ PDF downloaded via PMC fallback /pdf/ endpoint.
[2/12] PMID 31281883 — A heritable subset of the core rumen microbiome dictates dairy cow pro...
  ✓ tgz=True pdf=True nxml=False
[3/12] PMID 31418488 — Identification of rumen microbial biomarkers linked to methane emissio...
  ✓ tgz=True pdf=True nxml=False
  ⚠ PDF downloaded via PMC fallback /pdf/ endpoint.
[4/12] PMID 37491204 — Combining host and rumen metagenome profiling for selection in sheep: ...
  ✓ tgz=True pdf=True nxml=False
  ⚠ PDF downloaded via PMC fallback /pdf/ endpoint.
[5/12] PMID 37723422 — Large-scale analysis of sheep rumen metagenome profiles captured by re...
  ✓ tgz=True pdf=True nxml=False
  ⚠ PDF downloaded via PMC fallback /pdf/ endpoint.
[6/12] PMID 23739472 — Reducing GHG emissions through genetic improvement for feed efficien